In [1]:
# sacif_ppo.py
# ============================================================
# SACIF: Sleep-Adaptive Cognitive Interaction Framework
# - POMDP setting: hidden user state (sleep/capacity), partial observations (chat signals)
# - Agent learns a policy conditioned on belief features (mean/std/q10/q90 from particles)
#
# What you get:
# 1) SyntheticSleepUser (hidden dynamics)
# 2) Particle Belief Tracker (belief update from observations)
# 3) Gymnasium Env producing obs = [belief_summary(4) + raw_obs(6)] => 10-dim
# 4) PPO training (stable baseline) + evaluation
#
# Run:
#   python sacif_ppo.py
# Optional:
#   python sacif_ppo.py --steps 300000 --seed 0 --device cuda
# ============================================================

import argparse
import math
import random
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np

try:
    import gymnasium as gym
    from gymnasium import spaces
    GYMNASIUM = True
except ImportError:
    import gym
    from gym import spaces
    GYMNASIUM = False

import torch
import torch.nn as nn
import torch.optim as optim


# -----------------------------
# Utils
# -----------------------------
def clip(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))

def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# -----------------------------
# 1) Synthetic user (hidden state)
# -----------------------------
@dataclass
class UserState:
    t: int
    circadian: float      # [-1, 1]
    homeostatic: float    # [0, 1]
    sleep_debt: float     # [0, 2]
    capacity: float       # z in [0, 1] (hidden)
    mood: float           # [-1, 1]
    adherence: float      # [0, 1]


class SyntheticSleepUser:
    """
    Simplified Two-process + behavior coupling:
      - circadian: sine wave
      - homeostatic & sleep_debt: increase when awake, decrease when "did_sleep"
      - capacity z: sigmoid(linear combo + noise)
    Agent controls "dialogue burden" through actions, affecting adherence and sleep probability.
    """

    def __init__(self, seed: int = 0):
        self.rng = random.Random(seed)
        self.np_rng = np.random.default_rng(seed)
        self.reset()

    def reset(self) -> UserState:
        self.state = UserState(
            t=0,
            circadian=0.0,
            homeostatic=0.25,
            sleep_debt=0.45,
            capacity=0.70,
            mood=0.0,
            adherence=0.60,
        )
        return self.state

    def _update_physiology(self, did_sleep: bool):
        s = self.state
        # circadian (24 steps)
        s.circadian = math.sin(2.0 * math.pi * (s.t % 24) / 24.0)

        # homeostatic & sleep_debt
        if did_sleep:
            s.homeostatic = clip(s.homeostatic - (0.22 + 0.06 * self.rng.random()), 0.0, 1.0)
            s.sleep_debt  = clip(s.sleep_debt  - (0.18 + 0.06 * self.rng.random()), 0.0, 2.0)
        else:
            s.homeostatic = clip(s.homeostatic + (0.08 + 0.04 * self.rng.random()), 0.0, 1.0)
            s.sleep_debt  = clip(s.sleep_debt  + (0.06 + 0.04 * self.rng.random()), 0.0, 2.0)

        # mood drift (poor sleep -> negative)
        mood_drift = (-0.03 * s.sleep_debt + 0.02 * (1.0 - s.homeostatic) + self.np_rng.normal(0, 0.02))
        s.mood = clip(s.mood + mood_drift, -1.0, 1.0)

        # capacity z (hidden)
        raw = (
            1.1 * s.circadian
            - 2.1 * s.homeostatic
            - 0.9 * s.sleep_debt
            + 0.8 * s.mood
        )
        raw += float(self.np_rng.normal(0, 0.25))
        s.capacity = clip(sigmoid(raw), 0.0, 1.0)

    def step(self, action: int) -> Dict:
        """
        action ∈ {0,1,2,3} => dialogue burden
        modifies adherence & probability of 'did_sleep'
        """
        s = self.state
        burden = [0.10, 0.25, 0.45, 0.70][action]

        mismatch = abs(burden - s.capacity)
        adh_delta = 0.06 * (0.40 - mismatch) + float(self.np_rng.normal(0, 0.02))
        s.adherence = clip(s.adherence + adh_delta, 0.0, 1.0)

        p_sleep = clip(0.15 + 0.68 * s.adherence - 0.48 * max(0.0, burden - s.capacity), 0.0, 1.0)
        did_sleep = (self.rng.random() < p_sleep)

        s.t += 1
        self._update_physiology(did_sleep=did_sleep)

        return {
            "burden": burden,
            "did_sleep": did_sleep,
            "p_sleep": p_sleep,
            "true_capacity": s.capacity,   # for logging only
            "adherence": s.adherence,
            "mood": s.mood,
            "circadian": s.circadian,
            "homeostatic": s.homeostatic,
            "sleep_debt": s.sleep_debt,
        }

    def observe(self, action: int) -> np.ndarray:
        """
        Observation o_t from hidden state (agent sees this, not capacity directly)
        o = [latency, msg_len, sentiment, drop_prob, tod_sin, tod_cos]
        """
        s = self.state
        burden = [0.10, 0.25, 0.45, 0.70][action]

        latency = clip(
            0.8
            + 3.4 * (1.0 - s.capacity)
            + 2.2 * max(0.0, burden - s.capacity)
            + float(np.random.normal(0, 0.3)),
            0.2, 10.0
        )

        base_len = 120 + 240 * s.capacity
        penalty = 230 * max(0.0, burden - s.capacity)
        msg_len = clip(base_len - penalty + float(np.random.normal(0, 30)), 10, 500)

        sentiment = clip(s.mood - 0.8 * max(0.0, burden - s.capacity) + float(np.random.normal(0, 0.1)), -1.0, 1.0)

        drop_prob = clip(
            0.05 + 0.75 * max(0.0, burden - s.capacity) + 0.25 * (1.0 - s.adherence) + float(np.random.normal(0, 0.05)),
            0.0, 1.0
        )

        tod = (s.t % 24) / 24.0 * 2.0 * math.pi
        tod_sin, tod_cos = math.sin(tod), math.cos(tod)

        return np.array([latency, msg_len, sentiment, drop_prob, tod_sin, tod_cos], dtype=np.float32)


# -----------------------------
# 2) Particle Belief Tracker
# -----------------------------
class ParticleBelief:
    """
    Maintains particles for latent cognitive capacity z ∈ [0,1]
    and updates using a hand-crafted likelihood model.
    """

    def __init__(self, n_particles: int = 256, seed: int = 0):
        self.n = int(n_particles)
        self.rng = np.random.default_rng(seed)
        self.reset()

    def reset(self):
        self.z = self.rng.uniform(0.0, 1.0, size=self.n).astype(np.float32)
        self.w = np.ones(self.n, dtype=np.float32) / self.n

    def predict(self):
        # diffusion
        self.z = np.clip(self.z + self.rng.normal(0, 0.06, size=self.n).astype(np.float32), 0.0, 1.0)

    def update(self, obs: np.ndarray, action: int):
        latency, msg_len, sentiment, drop_prob = float(obs[0]), float(obs[1]), float(obs[2]), float(obs[3])
        burden = [0.10, 0.25, 0.45, 0.70][action]
        z = self.z

        exp_latency = 0.8 + 3.2 * (1.0 - z) + 2.0 * np.maximum(0.0, burden - z)
        exp_len     = 120 + 240 * z - 210 * np.maximum(0.0, burden - z)
        exp_sent    = 0.2 * (2.0 * z - 1.0) - 0.6 * np.maximum(0.0, burden - z)
        exp_drop    = 0.05 + 0.65 * np.maximum(0.0, burden - z)

        ll = (
            -0.5 * ((latency   - exp_latency) / 0.9)  ** 2
            -0.5 * ((msg_len   - exp_len)     / 60.0) ** 2
            -0.5 * ((sentiment - exp_sent)    / 0.25) ** 2
            -0.5 * ((drop_prob - exp_drop)    / 0.20) ** 2
        )

        ll = ll - np.max(ll)  # stabilize
        w = np.exp(ll).astype(np.float32)
        w = w / (np.sum(w) + 1e-8)
        self.w = w

        # resample
        idx = self.rng.choice(self.n, size=self.n, replace=True, p=self.w)
        self.z = self.z[idx]
        self.w = np.ones(self.n, dtype=np.float32) / self.n

    def summary(self) -> np.ndarray:
        mean = float(np.mean(self.z))
        std  = float(np.std(self.z))
        q10, q90 = np.quantile(self.z, [0.10, 0.90]).tolist()
        return np.array([mean, std, float(q10), float(q90)], dtype=np.float32)


# -----------------------------
# 3) Gym Env
# -----------------------------
class SACIFEnv(gym.Env):
    """
    Agent obs:
      - belief_summary(4): mean,std,q10,q90 of latent capacity
      - raw obs(6): latency,msg_len,sentiment,drop_prob,tod_sin,tod_cos
      => 10 dims
    Action:
      - 0..3 burden level
    Reward:
      - engagement (1-drop_prob)
      - adherence
      - load-match (1-|burden-capacity|)
    """

    metadata = {"render_modes": []}

    def __init__(self, seed: int = 0, horizon: int = 200):
        super().__init__()
        self.seed = int(seed)
        self.horizon = int(horizon)

        self.user = SyntheticSleepUser(seed=self.seed)
        self.belief = ParticleBelief(n_particles=256, seed=self.seed + 123)

        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(10,), dtype=np.float32)

        self.t = 0
        self.last_raw = np.zeros(6, dtype=np.float32)

    def reset(self, *, seed: Optional[int] = None, options=None):
        if seed is not None:
            self.seed = int(seed)
        self.user.reset()
        self.belief.reset()
        self.t = 0
        self.last_raw = np.zeros(6, dtype=np.float32)

        b = self.belief.summary()
        obs = np.concatenate([b, self.last_raw], axis=0).astype(np.float32)
        info = {}
        if GYMNASIUM:
            return obs, info
        return obs

    def step(self, action: int):
        self.t += 1

        self.belief.predict()
        raw = self.user.observe(action)
        self.last_raw = raw.copy()
        self.belief.update(raw, action)
        b = self.belief.summary()

        info_u = self.user.step(action)
        burden = info_u["burden"]
        capacity = info_u["true_capacity"]
        adherence = info_u["adherence"]

        drop_prob = float(raw[3])
        engagement = 1.0 - drop_prob
        match = 1.0 - abs(burden - capacity)

        reward = 0.45 * engagement + 0.35 * adherence + 0.20 * match

        terminated = (self.t >= self.horizon)
        truncated = False

        obs = np.concatenate([b, raw], axis=0).astype(np.float32)
        info = {
            "reward_engagement": engagement,
            "reward_match": match,
            "reward_adherence": adherence,
            **info_u
        }

        if GYMNASIUM:
            return obs, float(reward), terminated, truncated, info
        return obs, float(reward), terminated, info


# -----------------------------
# 4) PPO Actor-Critic
# -----------------------------
class ActorCritic(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh(),
        )
        self.pi = nn.Linear(128, n_actions)
        self.v  = nn.Linear(128, 1)

    def forward(self, x: torch.Tensor):
        h = self.body(x)
        logits = self.pi(h)
        value = self.v(h).squeeze(-1)
        return logits, value


@torch.no_grad()
def rollout(env: SACIFEnv, model: ActorCritic, steps: int, device: str, gamma: float, lam: float):
    """
    Collect on-policy rollout for PPO with GAE.
    Returns dict of tensors.
    """
    if GYMNASIUM:
        obs, _ = env.reset()
    else:
        obs = env.reset()

    obs_buf = []
    act_buf = []
    logp_buf = []
    val_buf = []
    rew_buf = []
    done_buf = []

    info_acc = {"engagement": [], "match": [], "adherence": [], "true_capacity": [], "burden": []}

    for _ in range(steps):
        x = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        logits, value = model(x)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        logp = dist.log_prob(action)

        a = int(action.item())
        if GYMNASIUM:
            obs2, r, terminated, truncated, info = env.step(a)
            done = terminated or truncated
        else:
            obs2, r, done, info = env.step(a)

        obs_buf.append(obs)
        act_buf.append(a)
        logp_buf.append(float(logp.item()))
        val_buf.append(float(value.item()))
        rew_buf.append(float(r))
        done_buf.append(float(done))

        info_acc["engagement"].append(float(info["reward_engagement"]))
        info_acc["match"].append(float(info["reward_match"]))
        info_acc["adherence"].append(float(info["reward_adherence"]))
        info_acc["true_capacity"].append(float(info["true_capacity"]))
        info_acc["burden"].append(float(info["burden"]))

        obs = obs2
        if done:
            if GYMNASIUM:
                obs, _ = env.reset()
            else:
                obs = env.reset()

    # compute GAE
    x = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
    _, last_v = model(x)
    last_v = float(last_v.item())

    adv = np.zeros(steps, dtype=np.float32)
    ret = np.zeros(steps, dtype=np.float32)

    gae = 0.0
    for t in reversed(range(steps)):
        next_v = last_v if t == steps - 1 else val_buf[t + 1]
        nonterminal = 1.0 - done_buf[t]
        delta = rew_buf[t] + gamma * next_v * nonterminal - val_buf[t]
        gae = delta + gamma * lam * nonterminal * gae
        adv[t] = gae

    ret = adv + np.array(val_buf, dtype=np.float32)

    batch = {
        "obs": torch.tensor(np.array(obs_buf, dtype=np.float32), device=device),
        "act": torch.tensor(np.array(act_buf, dtype=np.int64), device=device),
        "logp_old": torch.tensor(np.array(logp_buf, dtype=np.float32), device=device),
        "val_old": torch.tensor(np.array(val_buf, dtype=np.float32), device=device),
        "adv": torch.tensor(adv, device=device),
        "ret": torch.tensor(ret, device=device),
        "info": info_acc
    }
    return batch


def ppo_update(
    model: ActorCritic,
    opt: optim.Optimizer,
    batch: Dict,
    clip_ratio: float,
    vf_coef: float,
    ent_coef: float,
    train_iters: int,
    minibatch_size: int,
):
    obs = batch["obs"]
    act = batch["act"]
    logp_old = batch["logp_old"]
    adv = batch["adv"]
    ret = batch["ret"]

    # normalize advantage
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)

    n = obs.shape[0]
    idxs = np.arange(n)

    for _ in range(train_iters):
        np.random.shuffle(idxs)
        for start in range(0, n, minibatch_size):
            mb = idxs[start:start + minibatch_size]
            mb_obs = obs[mb]
            mb_act = act[mb]
            mb_logp_old = logp_old[mb]
            mb_adv = adv[mb]
            mb_ret = ret[mb]

            logits, value = model(mb_obs)
            dist = torch.distributions.Categorical(logits=logits)
            logp = dist.log_prob(mb_act)
            entropy = dist.entropy().mean()

            ratio = torch.exp(logp - mb_logp_old)
            surr1 = ratio * mb_adv
            surr2 = torch.clamp(ratio, 1.0 - clip_ratio, 1.0 + clip_ratio) * mb_adv
            pi_loss = -(torch.min(surr1, surr2)).mean()

            v_loss = 0.5 * (mb_ret - value).pow(2).mean()

            loss = pi_loss + vf_coef * v_loss - ent_coef * entropy

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()


@torch.no_grad()
def evaluate(env: SACIFEnv, model: ActorCritic, episodes: int, device: str) -> Dict[str, float]:
    totals = []
    em = []
    mm = []
    aa = []
    for _ in range(episodes):
        if GYMNASIUM:
            obs, _ = env.reset()
        else:
            obs = env.reset()

        done = False
        total = 0.0
        eng = 0.0
        mat = 0.0
        adh = 0.0
        steps = 0

        while not done:
            x = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            logits, _ = model(x)
            a = int(torch.argmax(logits, dim=-1).item())

            if GYMNASIUM:
                obs, r, terminated, truncated, info = env.step(a)
                done = terminated or truncated
            else:
                obs, r, done, info = env.step(a)

            total += float(r)
            eng += float(info["reward_engagement"])
            mat += float(info["reward_match"])
            adh += float(info["reward_adherence"])
            steps += 1

        totals.append(total / max(1, steps))
        em.append(eng / max(1, steps))
        mm.append(mat / max(1, steps))
        aa.append(adh / max(1, steps))

    return {
        "eval_avg_reward_per_step": float(np.mean(totals)),
        "eval_engagement": float(np.mean(em)),
        "eval_match": float(np.mean(mm)),
        "eval_adherence": float(np.mean(aa)),
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--horizon", type=int, default=200)

    parser.add_argument("--steps", type=int, default=200_000)          # total env steps
    parser.add_argument("--rollout", type=int, default=2048)           # steps per iteration
    parser.add_argument("--train-iters", type=int, default=10)
    parser.add_argument("--minibatch", type=int, default=256)

    parser.add_argument("--gamma", type=float, default=0.99)
    parser.add_argument("--lam", type=float, default=0.95)

    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--clip", type=float, default=0.2)
    parser.add_argument("--vf", type=float, default=0.5)
    parser.add_argument("--ent", type=float, default=0.01)

    parser.add_argument("--eval-every", type=int, default=10)
    parser.add_argument("--eval-episodes", type=int, default=10)
    args = parser.parse_args()

    set_global_seed(args.seed)

    env = SACIFEnv(seed=args.seed, horizon=args.horizon)
    eval_env = SACIFEnv(seed=args.seed + 999, horizon=args.horizon)

    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    device = args.device
    model = ActorCritic(obs_dim, n_actions).to(device)
    opt = optim.Adam(model.parameters(), lr=args.lr)

    total_steps = 0
    it = 0

    while total_steps < args.steps:
        it += 1
        batch = rollout(
            env=env,
            model=model,
            steps=args.rollout,
            device=device,
            gamma=args.gamma,
            lam=args.lam,
        )
        total_steps += args.rollout

        # logging (train batch stats)
        info = batch["info"]
        avg_eng = float(np.mean(info["engagement"]))
        avg_mat = float(np.mean(info["match"]))
        avg_adh = float(np.mean(info["adherence"]))
        avg_cap = float(np.mean(info["true_capacity"]))
        avg_bur = float(np.mean(info["burden"]))
        avg_rew = float((0.45 * avg_eng + 0.35 * avg_adh + 0.20 * avg_mat))

        ppo_update(
            model=model,
            opt=opt,
            batch=batch,
            clip_ratio=args.clip,
            vf_coef=args.vf,
            ent_coef=args.ent,
            train_iters=args.train_iters,
            minibatch_size=args.minibatch,
        )

        if it % args.eval_every == 0:
            ev = evaluate(eval_env, model, episodes=args.eval_episodes, device=device)
            print(
                f"[it {it:04d}] steps={total_steps:7d} "
                f"train(eng={avg_eng:.3f}, match={avg_mat:.3f}, adh={avg_adh:.3f}, cap={avg_cap:.3f}, bur={avg_bur:.3f}, approxR={avg_rew:.3f}) "
                f"eval(R={ev['eval_avg_reward_per_step']:.3f}, eng={ev['eval_engagement']:.3f}, match={ev['eval_match']:.3f}, adh={ev['eval_adherence']:.3f})"
            )

    # final eval
    ev = evaluate(eval_env, model, episodes=30, device=device)
    print("[final]", ev)

    # save
    torch.save({"model": model.state_dict(), "args": vars(args)}, "sacif_ppo.pt")
    print("saved: sacif_ppo.pt")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--seed SEED] [--device DEVICE]
                             [--horizon HORIZON] [--steps STEPS]
                             [--rollout ROLLOUT] [--train-iters TRAIN_ITERS]
                             [--minibatch MINIBATCH] [--gamma GAMMA]
                             [--lam LAM] [--lr LR] [--clip CLIP] [--vf VF]
                             [--ent ENT] [--eval-every EVAL_EVERY]
                             [--eval-episodes EVAL_EPISODES]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\82108\AppData\Roaming\jupyter\runtime\kernel-v3acde5af1a5cb0da6511995f3851618de463a0cda.json


SystemExit: 2

c:\Users\82108\anaconda3\dfcjsic\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
